In [ ]:
import json
import pandas as pd

path = "datasets/meta_Movies_and_TV.jsonl"

records = []
with open(path, "r", encoding="utf-8") as f:
    for line in f:
        records.append(json.loads(line))

df = pd.DataFrame(records)
print(df.shape)
print(df.head())

(748224, 16)
  main_category                                              title  \
0   Prime Video                                               Glee   
1   Prime Video                                One Perfect Wedding   
2   Movies & TV  How to Make Animatronic Characters - Organic M...   
3   Prime Video             Ode to Joy: Beethoven's Symphony No. 9   
4   Prime Video                      Ben 10: Alien Force (Classic)   

               subtitle  average_rating  rating_number  \
0  UnentitledUnentitled             4.7         2004.0   
1                   NaN             3.0            6.0   
2                   NaN             5.0            7.0   
3                   NaN             4.3           35.0   
4  UnentitledUnentitled             4.7          360.0   

                                      features  \
0  [IMDb 6.8, 2013, 22 episodes, X-Ray, TV-14]   
1            [IMDb 6.1, 1 h 27 min, 2021, ALL]   
2                                           []   
3                

Check the column names.

In [3]:
print(df.columns.tolist())

['main_category', 'title', 'subtitle', 'average_rating', 'rating_number', 'features', 'description', 'price', 'images', 'videos', 'store', 'categories', 'details', 'parent_asin', 'bought_together', 'author']


Check how many rows are null/empty.

In [7]:
df['main_category'].value_counts()

main_category
Movies & TV                  359953
Prime Video                  349118
Sports & Outdoors              1146
Books                           508
AMAZON FASHION                  214
Entertainment                   209
Digital Music                   154
Toys & Games                    117
Health & Personal Care           82
Amazon Home                      74
Video Games                      63
All Electronics                  59
Office Products                  30
Cell Phones & Accessories        27
Tools & Home Improvement         25
Computers                        16
Industrial & Scientific          15
Pet Supplies                     12
Home Audio & Theater             11
Software                          9
All Beauty                        9
Musical Instruments               8
Arts, Crafts & Sewing             8
Grocery                           7
Camera & Photo                    6
GPS & Navigation                  2
Automotive                        1
Collectible Co

In [4]:
print(df['categories'].isna().sum(), "missing out of", len(df))
print(df['categories'].apply(lambda x: len(x) == 0 if isinstance(x, list) else True).sum(), "empty lists")
print(df['categories'].head(10).tolist())

313988 missing out of 748224
314860 empty lists
[['Comedy', 'Drama', 'Arts, Entertainment, and Culture', 'Music Videos and Concerts'], ['Comedy', 'Drama', 'Romance'], ['Movies & TV', 'Genre for Featured Categories', 'Special Interests'], ['Documentary'], ['Science Fiction', 'Comedy', 'Animation', 'Drama'], ['Movies & TV', 'Genre for Featured Categories', 'Horror'], ['Horror', 'Atmospheric', 'Intense'], ['Movies & TV', 'Movies'], ['Movies & TV', 'Genre for Featured Categories', 'Music Videos & Concerts'], ['Movies & TV', 'Blu-ray', 'Movies']]


Load the rating dataset.

In [ ]:
ratings = pd.read_csv('datasets/Movies_and_TV.csv')

print(ratings.shape)
print(ratings.head())
print("Unique users:", ratings['user_id'].nunique())
print("Unique items:", ratings['parent_asin'].nunique())

(7441129, 4)
                        user_id parent_asin  rating      timestamp
0  AGXVBIUFLFGMVLATYXHJYL4A5Q7Q  B0002J58ME     5.0  1146713492000
1  AGXVBIUFLFGMVLATYXHJYL4A5Q7Q  B0091VXC54     5.0  1443550066000
2  AGXVBIUFLFGMVLATYXHJYL4A5Q7Q  B016JGYSRY     5.0  1445968640000
3  AGXVBIUFLFGMVLATYXHJYL4A5Q7Q  B00OI7J214     5.0  1446579637000
4  AGXVBIUFLFGMVLATYXHJYL4A5Q7Q  B01AMUSHAC     5.0  1476708291000
Unique users: 657203
Unique items: 197943


Check the overlap with metadata.

In [6]:
rated_items = set(ratings['parent_asin'].unique())
meta_items = set(df['parent_asin'].unique())

overlap = rated_items & meta_items
print("Rated items:", len(rated_items))
print("Metadata items:", len(meta_items))
print("Overlap:", len(overlap))
print("Coverage (% of rated items with metadata):", round(len(overlap) / len(rated_items) * 100, 2), "%")

Rated items: 197943
Metadata items: 748224
Overlap: 197943
Coverage (% of rated items with metadata): 100.0 %


Join the datasets.

In [8]:
# keep only relevant metadata columns
meta_small = df[['parent_asin', 'title', 'categories', 'average_rating', 'main_category']]

merged = ratings.merge(meta_small, on='parent_asin', how='left')
print(merged.shape)
print(merged.head())

(7441129, 8)
                        user_id parent_asin  rating      timestamp  \
0  AGXVBIUFLFGMVLATYXHJYL4A5Q7Q  B0002J58ME     5.0  1146713492000   
1  AGXVBIUFLFGMVLATYXHJYL4A5Q7Q  B0091VXC54     5.0  1443550066000   
2  AGXVBIUFLFGMVLATYXHJYL4A5Q7Q  B016JGYSRY     5.0  1445968640000   
3  AGXVBIUFLFGMVLATYXHJYL4A5Q7Q  B00OI7J214     5.0  1446579637000   
4  AGXVBIUFLFGMVLATYXHJYL4A5Q7Q  B01AMUSHAC     5.0  1476708291000   

                         title  \
0  10 Minute Solution: Pilates   
1                          NaN   
2                          NaN   
3                          NaN   
4                          NaN   

                                          categories  average_rating  \
0  [Movies & TV, Featured Categories, DVD, Exerci...             4.6   
1                                               None             4.6   
2                                               None             4.6   
3                                               None             4.6   
4

In [ ]:
#check the missingness of main_category
merged.groupby('main_category')['title'].apply(lambda x: x.isna().mean())

main_category
AMAZON FASHION               0.000000
All Beauty                   0.000000
All Electronics              0.000000
Amazon Home                  0.000000
Books                        0.000000
Cell Phones & Accessories    0.000000
Computers                    0.000000
Digital Music                0.000000
Health & Personal Care       0.000000
Industrial & Scientific      0.000000
Movies & TV                  0.000000
Office Products              0.000000
Prime Video                  0.888627
Software                     0.000000
Sports & Outdoors            0.000000
Tools & Home Improvement     0.000000
Toys & Games                 0.000000
Video Games                  0.000000
Name: title, dtype: float64

In [ ]:
#check the missingness in category
merged.groupby('main_category')['categories'].apply(lambda x: x.isna().mean())

main_category
AMAZON FASHION               0.000000
All Beauty                   0.000000
All Electronics              0.000000
Amazon Home                  0.000000
Books                        0.000000
Cell Phones & Accessories    0.000000
Computers                    0.000000
Digital Music                0.000000
Health & Personal Care       0.000000
Industrial & Scientific      0.000000
Movies & TV                  0.000000
Office Products              0.000000
Prime Video                  0.888627
Software                     0.000000
Sports & Outdoors            0.000000
Tools & Home Improvement     0.000000
Toys & Games                 0.000000
Video Games                  0.000000
Name: categories, dtype: float64

The following checks the standard EDA statistics for this dataset: sparsity, rating distribution, interaction counts per user/item, and a genre breakdown.

In [11]:
import pandas as pd

n_users = merged['user_id'].nunique()
n_items = merged['parent_asin'].nunique()
n_ratings = len(merged)

# 1. Sparsity
sparsity = 1 - (n_ratings / (n_users * n_items))
print("Sparsity:", round(sparsity, 6))

# 2. Rating distribution
print("\nRating distribution:")
print(merged['rating'].value_counts(normalize=True).sort_index())

# 3. Interactions per user / per item
print("\nInteractions per user:")
print(merged.groupby('user_id').size().describe())

print("\nInteractions per item:")
print(merged.groupby('parent_asin').size().describe())

# 4. Timestamp range
ts = pd.to_datetime(merged['timestamp'], unit='ms')
print("\nDate range:", ts.min(), "to", ts.max())

# 5. Genre/category breakdown (only rows with usable categories)
from collections import Counter
cat_counts = Counter()
for cats in merged['categories'].dropna():
    if isinstance(cats, list):
        cat_counts.update(cats)

print("\nTop 15 category labels:")
for cat, count in cat_counts.most_common(15):
    print(cat, count)

Sparsity: 0.999943

Rating distribution:
rating
1.0    0.063605
2.0    0.047934
3.0    0.088005
4.0    0.174817
5.0    0.625639
Name: proportion, dtype: float64

Interactions per user:
count    657203.000000
mean         11.322421
std          18.906311
min           5.000000
25%           6.000000
50%           7.000000
75%          11.000000
max        3079.000000
dtype: float64

Interactions per item:
count    197943.000000
mean         37.592282
std         153.725867
min           5.000000
25%           7.000000
50%          13.000000
75%          30.000000
max       19768.000000
dtype: float64

Date range: 1997-12-31 15:29:20 to 2023-09-07 23:09:16.149000

Top 15 category labels:
Movies & TV 3216017
Featured Categories 684961
Studio Specials 654855
Drama 616256
Blu-ray 581991
Genre for Featured Categories 548745
DVD 494320
Comedy 402668
Movies 382480
Action & Adventure 332943
Sony Pictures Home Entertainment 198941
All Sony Pictures Titles 195075
Science Fiction 180939
All Titles